# Hardware Signal Quality Corrections
## Implementación de correcciones críticas para HackRF

**Focus**: Frequency Offset, ADC Clipping, Spectral Leakage, I-Q Imbalance, Adjacent Channel Interference

*Nota: Excluye Noise Floor e IQ Calibration (ya implementados en DatasetFM-1-2.ipynb)*

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Importar el módulo de correcciones
sys.path.insert(0, r'D:\wnOs\wsp\CODE\work\PersonalStuff\GCPDS\ANE2\maintenance\qualityAssurance\modules')
import hwCorrections as hw

print(f"✅ hwCorrections module loaded from {hw.__file__}")

---
## Phase 1: Cargar datos (Raw o Processed Database)

In [ ]:
# Configuración
DATABASE_TYPE = "processed"  # "raw" o "processed"
FM_START_MHZ = 88.0
FM_END_MHZ = 108.0
TARGET_CENTER_MHZ = 101.1  # Frecuencia de sintonización actual

DATA_DIR = Path(
    r"D:\wnOs\wsp\CODE\work\PersonalStuff\GCPDS\ANE2\maintenance\qualityAssurance\database\Database-FM-10Nodes"
)

print(f"Database Type: {DATABASE_TYPE}")
print(f"Data Directory: {DATA_DIR}")
print(f"Target Frequency: {TARGET_CENTER_MHZ:.1f} MHz")
print(f"FM Band: {FM_START_MHZ:.1f} - {FM_END_MHZ:.1f} MHz")

In [ ]:
# Cargar datos de CSV (pueden ser raw o procesados)
import glob

csv_files = sorted(glob.glob(str(DATA_DIR / "DataBase-RF-FM-88MHz-108MHz-Bogota-Funza" / "Node*.csv")))
print(f"Found {len(csv_files)} node CSV files")

# Cargar primeros 2 nodos como ejemplo
datos_nodos = {}
for csv_file in csv_files[:2]:
    node_name = Path(csv_file).stem
    try:
        df = pd.read_csv(csv_file, nrows=10)  # Primeras 10 filas para prueba rápida
        datos_nodos[node_name] = df
        print(f"✅ {node_name}: {len(df)} rows x {len(df.columns)} cols")
    except Exception as e:
        print(f"⚠️  {node_name}: {e}")

---
## Phase 2: ISSUE #1 - Frequency Offset Detection

In [ ]:
# Helper para extraer PSD del CSV
def parse_pxx_cell(pxx_raw):
    """Parsear PSD desde celda CSV (puede ser string, array, etc)"""
    if isinstance(pxx_raw, np.ndarray):
        return pxx_raw.astype(float)
    if isinstance(pxx_raw, (list, tuple)):
        return np.array(pxx_raw, dtype=float)
    
    s = str(pxx_raw).strip()
    if s.startswith('[') and s.endswith(']'):
        try:
            parts = s[1:-1].replace(',', ' ').split()
            return np.fromiter(map(float, parts), dtype=float)
        except:
            pass
    
    try:
        import ast
        return np.asarray(ast.literal_eval(s), dtype=float).ravel()
    except:
        return np.array([])

# Analizar frequency offset para cada nodo
freq_offset_results = {}

for node_name, df in datos_nodos.items():
    print(f"\n{'='*80}")
    print(f"Analyzing: {node_name}")
    print(f"{'='*80}")
    
    if 'pxx' not in df.columns:
        print(f"⚠️  'pxx' column not found")
        continue
    
    # Tomar primera fila válida
    for row_idx in range(len(df)):
        pxx = parse_pxx_cell(df['pxx'].iloc[row_idx])
        if pxx.size > 100:
            break
    
    if pxx.size < 100:
        print(f"⚠️  No valid PSD data")
        continue
    
    # Crear eje de frecuencias
    freqs_mhz = np.linspace(FM_START_MHZ, FM_END_MHZ, pxx.size)
    freqs_hz = freqs_mhz * 1e6
    target_hz = TARGET_CENTER_MHZ * 1e6
    
    # Detectar frequency offset usando 3 métodos
    methods = ["peak_tracking", "zero_crossing", "correlation"]
    
    for method in methods:
        try:
            result = hw.detect_frequency_offset(
                freqs_hz, pxx,
                target_freq_hz=target_hz,
                method=method,
            )
            
            status = "✅ OK" if result.is_within_specs else "⚠️  OUT OF SPEC"
            print(f"\n  [{method:15s}]")
            print(f"    Offset: {result.estimated_offset_hz/1e3:+.2f} kHz {status}")
            print(f"    Confidence: {result.confidence:.2%}")
            print(f"    Correction Factor: {result.correction_factor:.6f}")
            
            if node_name not in freq_offset_results:
                freq_offset_results[node_name] = {}
            freq_offset_results[node_name][method] = result
            
        except Exception as e:
            print(f"    Error: {e}")

---
## Phase 3: ISSUE #2 - ADC Clipping Detection

In [ ]:
# Análisis de clipping del ADC
adc_clipping_results = {}

for node_name, df in datos_nodos.items():
    print(f"\n{'='*80}")
    print(f"ADC Clipping Analysis: {node_name}")
    print(f"{'='*80}")
    
    if 'pxx' not in df.columns:
        continue
    
    # Usar mismo PSD que arriba
    for row_idx in range(len(df)):
        pxx = parse_pxx_cell(df['pxx'].iloc[row_idx])
        if pxx.size > 100:
            break
    
    if pxx.size < 100:
        continue
    
    # Detectar clipping usando PSD
    clipping_result = hw.detect_adc_clipping(
        pxx_db=pxx,
        adc_ceiling_db=0.0,  # Típico para HackRF normalizado
        adc_headroom_target_db=6.0,
    )
    
    print(f"\n  Severity Level: {clipping_result.severity_level.upper()}")
    print(f"  Clipping Detected: {clipping_result.clipping_detected}")
    print(f"  Clipping %: {clipping_result.clipping_percentage:.2f}%")
    print(f"  Headroom: {clipping_result.headroom_db:+.2f} dB")
    print(f"  Action: {clipping_result.recommended_action}")
    
    adc_clipping_results[node_name] = clipping_result

---
## Phase 4: ISSUE #3 - Spectral Leakage Estimation

In [ ]:
# Análisis de leakage espectral
leakage_results = {}

for node_name, df in datos_nodos.items():
    print(f"\n{'='*80}")
    print(f"Spectral Leakage Analysis: {node_name}")
    print(f"{'='*80}")
    
    if 'pxx' not in df.columns:
        continue
    
    # Usar mismo PSD
    for row_idx in range(len(df)):
        pxx = parse_pxx_cell(df['pxx'].iloc[row_idx])
        if pxx.size > 100:
            break
    
    if pxx.size < 100:
        continue
    
    freqs_mhz = np.linspace(FM_START_MHZ, FM_END_MHZ, pxx.size)
    freqs_hz = freqs_mhz * 1e6
    target_hz = TARGET_CENTER_MHZ * 1e6
    
    # Estimar leakage
    leakage_result, corrected_pxx = hw.estimate_spectral_leakage(
        pxx, freqs_hz,
        tone_freq_hz=target_hz,
        window_type="hann",
    )
    
    print(f"\n  Leakage Power: {leakage_result.leakage_power_db:.2f} dB")
    print(f"  Leakage Ratio: {leakage_result.leakage_ratio:.4f} ({leakage_result.leakage_ratio*100:.2f}%)")
    print(f"  Spectral Efficiency: {leakage_result.spectral_efficiency:.4f}")
    print(f"  Current Window: hann")
    print(f"  Recommended Window: {leakage_result.recommended_window}")
    
    leakage_results[node_name] = leakage_result

---
## Phase 5: ISSUE #4 - I-Q Imbalance Detection

In [ ]:
# Para I-Q imbalance, necesitamos muestras IQ complejas (no solo PSD)
# Aquí mostramos cómo hacerlo SI los datos IQ están disponibles

print("\n" + "="*80)
print("I-Q IMBALANCE DETECTION")
print("="*80)

print("""
📌 NOTA: Para detección de I-Q imbalance se requieren muestras IQ complejas.


Codigo de ejemplo (si tienes muestras IQ):

```python
# Ejemplo de uso con muestras IQ
if 'iq_samples' in df.columns:  # O similar, según tu estructura de datos
    for node_name, df in datos_nodos.items():
        # Extraer muestras IQ
        iq_raw = df['iq_samples'].iloc[0]
        iq_samples = parse_iq_cell(iq_raw)  # Función para parsear IQ
        
        # Detectar imbalance
        iq_result = hw.detect_iq_imbalance(iq_samples)
        
        print(f"{node_name}:")
        print(f"  Amplitude Imbalance: {iq_result.amplitude_imbalance_db:+.2f} dB")
        print(f"  Phase Imbalance: {iq_result.phase_imbalance_deg:+.2f}°")
        print(f"  Severity: {iq_result.severity}")
        print(f"  Correction Coefficients: {iq_result.correction_coefficients}")
```
""")

---
## Phase 6: ISSUE #5 - Adjacent Channel Interference (ACI)

In [ ]:
# Detección de interferencia de canales adyacentes
aci_results = {}

for node_name, df in datos_nodos.items():
    print(f"\n{'='*80}")
    print(f"Adjacent Channel Interference Analysis: {node_name}")
    print(f"{'='*80}")
    
    if 'pxx' not in df.columns:
        continue
    
    # Usar mismo PSD
    for row_idx in range(len(df)):
        pxx = parse_pxx_cell(df['pxx'].iloc[row_idx])
        if pxx.size > 100:
            break
    
    if pxx.size < 100:
        continue
    
    freqs_mhz = np.linspace(FM_START_MHZ, FM_END_MHZ, pxx.size)
    freqs_hz = freqs_mhz * 1e6
    target_hz = TARGET_CENTER_MHZ * 1e6
    
    # Detectar ACI (FM: 200 kHz spacing, 200 kHz BW)
    aci_result = hw.detect_adjacent_channel_interference(
        pxx, freqs_hz,
        center_freq_hz=target_hz,
        channel_bw_hz=200e3,  # FM standard: 200 kHz
    )
    
    print(f"\n  ACI Power: {aci_result.aci_power_db:.2f} dB")
    print(f"  ACI Ratio: {aci_result.aci_ratio:.4f} ({aci_result.aci_ratio*100:.2f}%)")
    print(f"  Affected Bands: {aci_result.affected_bands if aci_result.affected_bands else 'None detected'}")
    print(f"  Recommended Cutoff Filter: {aci_result.mitigation_cuttoff_hz/1e3:.1f} kHz")
    
    if aci_result.aci_ratio > 0.05:
        print(f"  ⚠️  HIGH ACI — Consider narrower filter or improved selectivity")
    
    aci_results[node_name] = aci_result

---
## Phase 7: Summary Report

In [ ]:
# Crear tabla resumen
summary_data = []

for node_name in datos_nodos.keys():
    # Frequency Offset (best method)
    if node_name in freq_offset_results:
        freq_offset = freq_offset_results[node_name]['peak_tracking']
        freq_offset_khz = freq_offset.estimated_offset_hz / 1e3
    else:
        freq_offset_khz = None
    
    # ADC
    if node_name in adc_clipping_results:
        adc = adc_clipping_results[node_name]
        adc_severity = adc.severity_level
        adc_headroom = adc.headroom_db
    else:
        adc_severity = None
        adc_headroom = None
    
    # Leakage
    if node_name in leakage_results:
        leakage = leakage_results[node_name]
        leakage_pct = leakage.leakage_ratio * 100
        leakage_window = leakage.recommended_window
    else:
        leakage_pct = None
        leakage_window = None
    
    # ACI
    if node_name in aci_results:
        aci = aci_results[node_name]
        aci_pct = aci.aci_ratio * 100
    else:
        aci_pct = None
    
    summary_data.append({
        'Node': node_name,
        'Freq Offset (kHz)': f"{freq_offset_khz:+.2f}" if freq_offset_khz is not None else "N/A",
        'ADC Severity': adc_severity if adc_severity else "N/A",
        'Headroom (dB)': f"{adc_headroom:.2f}" if adc_headroom is not None else "N/A",
        'Leakage (%)': f"{leakage_pct:.2f}" if leakage_pct is not None else "N/A",
        'ACI (%)': f"{aci_pct:.2f}" if aci_pct is not None else "N/A",
    })

df_summary = pd.DataFrame(summary_data)
print("\n" + "="*140)
print("📊 HARDWARE QUALITY SUMMARY")
print("="*140)
print(df_summary.to_string(index=False))
print("\nNote: Rows show key metrics for each node. N/A = insufficient data.")

---
## Next Steps

### 1️⃣ Frequency Offset Correction
- Use `correction_factor` to adjust future tuning
- Apply offset tracking between measurement sessions

### 2️⃣ ADC Clipping Mitigation
- If severity = "moderate" or "severe": reduce RF gain
- Recalibrate gain settings per node

### 3️⃣ Spectral Leakage
- Change FFT window if `recommended_window` differs from current
- Rebuild PSD with improved window function

### 4️⃣ Adjacent Channel Interference
- If ACI > 5%: implement narrower BPF
- Check for strong adjacent FM stations

### 5️⃣ I-Q Imbalance (if IQ data available)
- Apply correction using `correction_coefficients`
- Store per-node correction matrix for real-time correction